### Redrob Hackathon: Hybrid Retrieval Baseline

## What this notebook does
This notebook retrieves the most plausible candidates for the Senior AI Engineer role from the full 100,000-candidate pool.

## Why this notebook is needed
Ranking every candidate deeply from the start is inefficient and can introduce noise.

We first retrieve a smaller shortlist using:
1. BM25 keyword retrieval
2. Dense semantic retrieval
3. Reciprocal Rank Fusion

The shortlisted candidates will be passed to the reranking and trap-filtering stage.

In [1]:
!pip install -q \
rank-bm25 \
sentence-transformers \
faiss-cpu \
pandas \
numpy \
tqdm \
pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 102.2 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
import re

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
PROJECT_DIR = Path("/content/drive/MyDrive/Redrob_Hackathon")
DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"

FEATURES_PATH = ARTIFACTS_DIR / "candidate_features.parquet"
JD_TEXT_PATH = ARTIFACTS_DIR / "job_description_cleaned.txt"
JD_COMPASS_PATH = ARTIFACTS_DIR / "jd_compass.json"

In [5]:
print("Features file exists:", FEATURES_PATH.exists())
print("JD text file exists:", JD_TEXT_PATH.exists())
print("JD compass file exists:", JD_COMPASS_PATH.exists())

Features file exists: True
JD text file exists: True
JD compass file exists: True


In [6]:
features_df = pd.read_parquet(FEATURES_PATH)
features_df.head()

,candidate_id,anonymized_name,headline,summary,location,country,years_of_experience,current_title,current_company,current_company_size,...,availability_strength,core_ai_ml_match_count,retrieval_ranking_match_count,production_engineering_match_count,evaluation_quality_match_count,seniority_fit_score,location_fit_flag,flag_possible_framework_only_profile,flag_possible_non_ir_ai_profile,role_evidence_score
0,CAND_0000001,Ira Vora,"Backend Engineer | SQL, Spark, Cloud",Software / data professional with 6.9 years of...,Toronto,Canada,6.9,Backend Engineer,Mindtree,10001+,...,31.019672,0,1,1,0,1.0,0,False,False,6.5
1,CAND_0000002,Saanvi Sethi,Operations Manager | 12.5+ yrs experience,Professional with 12.5+ years of experience. M...,"Chennai, Tamil Nadu",India,12.5,Operations Manager,Wipro,10001+,...,29.519672,1,0,1,0,0.0,0,False,False,2.5
2,CAND_0000003,Yash Agarwal,Customer Support | 1.1+ yrs experience,Professional with 1.1+ years of experience. I'...,Austin,USA,1.1,Customer Support,TCS,10001+,...,14.131126,0,0,0,0,0.0,0,False,False,0.0
3,CAND_0000004,Anil Bose,Marketing Manager | Driving business outcomes,Professional with 3.8+ years of experience. My...,Sydney,Australia,3.8,Marketing Manager,Dunder Mifflin,201-500,...,8.213223,1,0,1,0,0.3,0,False,False,3.4
4,CAND_0000005,Aisha Sethi,Accountant | Helping teams scale,Professional with 11.0+ years of experience. I...,"Gurgaon, Haryana",India,11.0,Accountant,Stark Industries,1001-5000,...,32.712903,0,0,0,0,0.3,1,False,False,1.4


In [7]:
jd_text = JD_TEXT_PATH.read_text(encoding = 'utf-8')
jd_text[:1000]

'Job Description: Senior AI Engineer — Founding Team\nCompany: Redrob AI (Series A AI-native talent intelligence platform)\nLocation: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities\nEmployment Type: Full-time\nExperience Required: 5–9 years (see "what we mean by this" below)\nLet\'s be honest about this role\nWe\'re going to write this JD differently from most. We\'re a Series A company that just raised our round and we\'re building a new AI Engineering org from scratch. This is the kind of role where the JD changes every six months because the company changes every six months. So instead of pretending we have a fixed checklist, we\'re going to tell you what we actually need and what we\'ve gotten wrong before.\nIf you\'ve spent your career at Google or Meta and you want a well-scoped role with a defined ladder, this isn\'t it.\nIf you\'ve spent your career bouncing between early-stage startups and you want to "just code" without

In [8]:
with open(JD_COMPASS_PATH, 'r', encoding = 'utf-8') as f:
    jd_compass = json.load(f)
print("JD role:", jd_compass["jd_summary"]["role_title"])

JD role: Senior AI Engineer


In [9]:
def clean_text(value):
  if value is None:
    return ""
  value = str(value)
  value = value.strip()
  value = re.sub(r"\s+", " ", value)
  return value

features_df["retrieval_text"] = (
    "Current title: " + features_df["current_title"].fillna("") + ". "
    + "Headline: " + features_df["headline"].fillna("") + ". "
    + "Summary: " + features_df["summary"].fillna("") + ". "
    + "Skills: " + features_df["skill_text"].fillna("") + ". "
    + "Career history: " + features_df["profile_text"].fillna("")
)

retrieval_texts = features_df["retrieval_text"].tolist()

print("Total retrieval documents:", len(retrieval_texts))
print("\nExample retrieval document:\n")
print(retrieval_texts[0][:1500])

Total retrieval documents: 100000

Example retrieval document:

Current title: Backend Engineer. Headline: Backend Engineer | SQL, Spark, Cloud. Summary: Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.. Skills: Tailwind | NLP | Image Classification | Fine-tuning LLMs | Weights & Biases | Speech Recognition | Photoshop | TTS | LoRA | Apache Beam | AWS | Flask | BentoML | Milvus | GANs | Statistical Modeling | G

In [10]:
retrieval_query = """
Senior AI Engineer with production experience in machine learning, embeddings,
semantic search, information retrieval, hybrid search, ranking, recommendation
systems, vector databases, FAISS, Elasticsearch, OpenSearch, Pinecone, Qdrant,
evaluation frameworks, NDCG, MRR, MAP, A/B testing, Python, deployment,
MLOps, APIs, Docker, cloud infrastructure, LLMs, retrieval systems,
search systems, matching systems, and real product engineering.
"""

print(retrieval_query)


Senior AI Engineer with production experience in machine learning, embeddings,
semantic search, information retrieval, hybrid search, ranking, recommendation
systems, vector databases, FAISS, Elasticsearch, OpenSearch, Pinecone, Qdrant,
evaluation frameworks, NDCG, MRR, MAP, A/B testing, Python, deployment,
MLOps, APIs, Docker, cloud infrastructure, LLMs, retrieval systems,
search systems, matching systems, and real product engineering.



In [11]:
# BM25 preprocessing
def tokenize(text):
    return clean_text(text).lower().split()

tokenized_corpus = []

for text in tqdm(retrieval_texts):
    tokens = tokenize(text)
    tokenized_corpus.append(tokens)
tokenized_query = tokenize(retrieval_query)

print("Example tokenized candidate text:")
print(tokenized_corpus[0][:40])

  0%|          | 0/100000 [00:00<?, ?it/s]

Example tokenized candidate text:
['current', 'title:', 'backend', 'engineer.', 'headline:', 'backend', 'engineer', '|', 'sql,', 'spark,', 'cloud.', 'summary:', 'software', '/', 'data', 'professional', 'with', '6.9', 'years', 'of', 'experience', 'building', 'data', 'pipelines,', 'backend', 'systems,', 'and', 'analytics', 'infrastructure.', "i'm", 'a', 'backend/data', 'hybrid', '—', 'spark,', 'airflow,', 'sql', 'warehouses', 'are', 'home']


In [12]:
bm25 = BM25Okapi(tokenized_corpus)

bm25_scores = bm25.get_scores(tokenized_query)

features_df["bm25_score"] = bm25_scores

bm25_top_k = 2000

bm25_top_indices = np.argsort(bm25_scores)[::-1][:bm25_top_k]

bm25_results = features_df.iloc[bm25_top_indices][
    [
        "candidate_id",
        "current_title",
        "current_company",
        "years_of_experience",
        "bm25_score",
        "role_evidence_score",
        "total_risk_score",
    ]
].copy()

display(bm25_results.head(20))

,candidate_id,current_title,current_company,years_of_experience,bm25_score,role_evidence_score,total_risk_score
81845,CAND_0081846,Lead AI Engineer,Razorpay,6.7,133.914668,31.5,2
18498,CAND_0018499,Senior Machine Learning Engineer,Zomato,7.2,120.887798,25.0,0
46063,CAND_0046064,Senior NLP Engineer,Salesforce,8.9,120.143527,25.5,0
55904,CAND_0055905,Senior Machine Learning Engineer,Flipkart,8.1,115.497883,31.5,1
76903,CAND_0076904,Senior Data Scientist,Nykaa,4.2,114.699111,22.8,3
57700,CAND_0057701,Recommendation Systems Engineer,Verloop.io,4.1,113.642371,20.8,1
41567,CAND_0041568,Search Engineer,Haptik,5.2,113.634935,28.0,3
46524,CAND_0046525,Senior Machine Learning Engineer,Genpact AI,6.1,113.255306,27.0,0
5259,CAND_0005260,Senior NLP Engineer,Netflix,5.2,113.202236,25.5,3
39753,CAND_0039754,Senior Applied Scientist,Meta,16.2,111.393341,25.5,2


In [13]:
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
candidate_embeddings = embedding_model.encode(
    retrieval_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

candidate_embeddings = np.asarray(candidate_embeddings, dtype="float32")  # Because FAISS requires float32 objects only
print("Candidate embedding shape:", candidate_embeddings.shape)

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Candidate embedding shape: (100000, 384)


In [15]:
embedding_dimension = candidate_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dimension)

faiss_index.add(candidate_embeddings)

print("Vectors in FAISS index:", faiss_index.ntotal)

Vectors in FAISS index: 100000


In [16]:
query_embedding = embedding_model.encode(
    [retrieval_query],
    normalize_embeddings=True
)

query_embedding = np.asarray(query_embedding, dtype="float32")

dense_top_k = 2000

dense_scores, dense_indices = faiss_index.search(
    query_embedding,
    dense_top_k
)

dense_scores = dense_scores[0]
dense_indices = dense_indices[0]

dense_results = features_df.iloc[dense_indices][
    [
        "candidate_id",
        "current_title",
        "current_company",
        "years_of_experience",
        "role_evidence_score",
        "total_risk_score",
    ]
].copy()

dense_results["dense_score"] = dense_scores

display(dense_results.head(20))

,candidate_id,current_title,current_company,years_of_experience,role_evidence_score,total_risk_score,dense_score
2024,CAND_0002025,Senior AI Engineer,Apple,5.9,18.5,2,0.878832
62246,CAND_0062247,AI Engineer,Google,7.3,20.0,0,0.878384
55991,CAND_0055992,AI Engineer,CRED,16.9,27.0,0,0.877744
46524,CAND_0046525,Senior Machine Learning Engineer,Genpact AI,6.1,27.0,0,0.876455
42505,CAND_0042506,Search Engineer,Verloop.io,4.2,25.3,2,0.875599
49895,CAND_0049896,Search Engineer,Unacademy,7.3,21.0,1,0.875365
41567,CAND_0041568,Search Engineer,Haptik,5.2,28.0,3,0.874794
24619,CAND_0024620,AI Engineer,PharmEasy,5.9,15.0,0,0.874469
76250,CAND_0076251,Search Engineer,Haptik,7.6,24.0,1,0.873893
93546,CAND_0093547,Senior Machine Learning Engineer,PhonePe,2.9,10.5,2,0.872618


In [17]:
bm25_ids = set(features_df.iloc[bm25_top_indices]["candidate_id"])
dense_ids = set(features_df.iloc[dense_indices]["candidate_id"])

overlap_ids = bm25_ids.intersection(dense_ids)

print("BM25 top candidates:", len(bm25_ids))
print("Dense top candidates:", len(dense_ids))
print("Overlap:", len(overlap_ids))
print("Overlap percentage:", round(len(overlap_ids) / 2000 * 100, 2), "%")

BM25 top candidates: 2000
Dense top candidates: 2000
Overlap: 951
Overlap percentage: 47.55 %


In [18]:
def reciprocal_rank_fusion(rankings, k=60):
    fused_scores = {}

    for ranking in rankings:
        for rank, candidate_id in enumerate(ranking, start=1):
            fused_scores[candidate_id] = fused_scores.get(candidate_id, 0) + 1 / (k + rank)

    return fused_scores

In [19]:
bm25_ranked_ids = features_df.iloc[bm25_top_indices]["candidate_id"].tolist()
dense_ranked_ids = features_df.iloc[dense_indices]["candidate_id"].tolist()

rrf_scores = reciprocal_rank_fusion(
    [bm25_ranked_ids, dense_ranked_ids],
    k=60
)

rrf_df = pd.DataFrame(
    list(rrf_scores.items()),
    columns=["candidate_id", "rrf_score"]
)

retrieval_df = features_df.merge(
    rrf_df,
    on="candidate_id",
    how="inner"
)

retrieval_df = retrieval_df.sort_values(
    "rrf_score",
    ascending=False
).reset_index(drop=True)

print("Candidates in fused retrieval pool:", retrieval_df.shape[0])

display(
    retrieval_df[
        [
            "candidate_id",
            "current_title",
            "current_company",
            "years_of_experience",
            "bm25_score",
            "role_evidence_score",
            "total_risk_score",
            "rrf_score",
        ]
    ].head(30)
)

Candidates in fused retrieval pool: 3049


,candidate_id,current_title,current_company,years_of_experience,bm25_score,role_evidence_score,total_risk_score,rrf_score
0,CAND_0046525,Senior Machine Learning Engineer,Genpact AI,6.1,113.255306,27.0,0,0.030331
1,CAND_0041568,Search Engineer,Haptik,5.2,113.634935,28.0,3,0.029851
2,CAND_0042506,Search Engineer,Verloop.io,4.2,107.670382,25.3,2,0.028898
3,CAND_0076904,Senior Data Scientist,Nykaa,4.2,114.699111,22.8,3,0.027885
4,CAND_0049896,Search Engineer,Unacademy,7.3,95.721426,21.0,1,0.026779
5,CAND_0046064,Senior NLP Engineer,Salesforce,8.9,120.143527,25.5,0,0.025774
6,CAND_0005649,Senior Data Scientist,Sarvam AI,7.4,111.334095,32.5,0,0.025579
7,CAND_0018499,Senior Machine Learning Engineer,Zomato,7.2,120.887798,25.0,0,0.025388
8,CAND_0055992,AI Engineer,CRED,16.9,87.197884,27.0,0,0.024882
9,CAND_0042029,Senior Data Scientist,Flipkart,6.5,101.505897,24.5,2,0.024706


In [20]:
retrieval_df["retrieval_guardrail_score"] = (
    retrieval_df["role_evidence_score"] * 0.25
    + retrieval_df["availability_strength"] * 0.10
    - retrieval_df["total_risk_score"] * 0.15
)

retrieval_df["retrieval_stage_score"] = (
    retrieval_df["rrf_score"] * 100
    + retrieval_df["retrieval_guardrail_score"]
)

retrieval_df = retrieval_df.sort_values(
    "retrieval_stage_score",
    ascending=False
).reset_index(drop=True)

display(
    retrieval_df[
        [
            "candidate_id",
            "current_title",
            "years_of_experience",
            "rrf_score",
            "role_evidence_score",
            "availability_strength",
            "total_risk_score",
            "retrieval_stage_score",
        ]
    ].head(30)
)

,candidate_id,current_title,years_of_experience,rrf_score,role_evidence_score,availability_strength,total_risk_score,retrieval_stage_score
0,CAND_0086022,Senior Applied Scientist,5.3,0.019737,27.5,86.500000,2,17.198684
1,CAND_0055905,Senior Machine Learning Engineer,8.1,0.022336,31.5,47.712903,1,14.729931
2,CAND_0046525,Senior Machine Learning Engineer,6.1,0.030331,27.0,47.219672,0,14.505055
3,CAND_0005649,Senior Data Scientist,7.4,0.025579,32.5,37.649451,0,14.447821
4,CAND_0069905,Applied ML Engineer,6.6,0.015768,32.0,43.949451,0,13.971742
5,CAND_0079387,AI Engineer,6.9,0.016299,32.0,45.912903,2,13.921192
6,CAND_0081846,Lead AI Engineer,6.7,0.019011,31.5,43.512903,2,13.827415
7,CAND_0055992,AI Engineer,16.9,0.024882,27.0,42.419672,0,13.480170
8,CAND_0046064,Senior NLP Engineer,8.9,0.025774,25.5,45.012903,0,13.453691
9,CAND_0040887,Machine Learning Engineer,4.7,0.019192,26.8,48.325000,0,13.451692


In [21]:
SHORTLIST_SIZE = 1500

shortlist_df = retrieval_df.head(SHORTLIST_SIZE).copy()

print("Final shortlist size:", shortlist_df.shape[0])

display(
    shortlist_df[
        [
            "candidate_id",
            "current_title",
            "current_company",
            "years_of_experience",
            "retrieval_stage_score",
            "total_risk_score",
        ]
    ].head(20)
)

Final shortlist size: 1500


,candidate_id,current_title,current_company,years_of_experience,retrieval_stage_score,total_risk_score
0,CAND_0086022,Senior Applied Scientist,Sarvam AI,5.3,17.198684,2
1,CAND_0055905,Senior Machine Learning Engineer,Flipkart,8.1,14.729931,1
2,CAND_0046525,Senior Machine Learning Engineer,Genpact AI,6.1,14.505055,0
3,CAND_0005649,Senior Data Scientist,Sarvam AI,7.4,14.447821,0
4,CAND_0069905,Applied ML Engineer,Sarvam AI,6.6,13.971742,0
5,CAND_0079387,AI Engineer,Microsoft,6.9,13.921192,2
6,CAND_0081846,Lead AI Engineer,Razorpay,6.7,13.827415,2
7,CAND_0055992,AI Engineer,CRED,16.9,13.480170,0
8,CAND_0046064,Senior NLP Engineer,Salesforce,8.9,13.453691,0
9,CAND_0040887,Machine Learning Engineer,Netflix,4.7,13.451692,0


In [22]:
RETRIEVAL_PATH = ARTIFACTS_DIR / "candidate_retrieval_shortlist.parquet"
RETRIEVAL_CSV_PATH = ARTIFACTS_DIR / "candidate_retrieval_shortlist.csv"

shortlist_df.to_parquet(RETRIEVAL_PATH, index=False)
shortlist_df.to_csv(RETRIEVAL_CSV_PATH, index=False)

print("Saved:", RETRIEVAL_PATH)
print("Saved:", RETRIEVAL_CSV_PATH)

Saved: /content/drive/MyDrive/Redrob_Hackathon/artifacts/candidate_retrieval_shortlist.parquet
Saved: /content/drive/MyDrive/Redrob_Hackathon/artifacts/candidate_retrieval_shortlist.csv


In [23]:
retrieval_metadata = {
    "total_candidates": int(features_df.shape[0]),
    "bm25_top_k": int(bm25_top_k),
    "dense_top_k": int(dense_top_k),
    "fused_pool_size": int(retrieval_df.shape[0]),
    "final_shortlist_size": int(shortlist_df.shape[0]),
    "embedding_model": "BAAI/bge-small-en-v1.5",
    "fusion_method": "Reciprocal Rank Fusion",
}

with open(ARTIFACTS_DIR / "retrieval_metadata.json", "w", encoding="utf-8") as f:
    json.dump(retrieval_metadata, f, indent=4)

print(retrieval_metadata)

{'total_candidates': 100000, 'bm25_top_k': 2000, 'dense_top_k': 2000, 'fused_pool_size': 3049, 'final_shortlist_size': 1500, 'embedding_model': 'BAAI/bge-small-en-v1.5', 'fusion_method': 'Reciprocal Rank Fusion'}
